# Ziffernerkennung – Logistische Regression mit MNIST

In diesem Notebook lernen wir die **logistische Regression** kennen – eine der wichtigsten Methoden im Machine Learning. Damit können wir **handgeschriebene Ziffern** (0–9) erkennen.

Der Plan:
1. **Logistische Regression** verstehen – was ist das und wie funktioniert sie?
2. **MNIST-Datensatz** laden – 70.000 handgeschriebene Ziffern als Trainingsdaten
3. **Modell trainieren** – den Classifier mit scikit-learn aufbauen
4. **Eigene Ziffern erkennen** – mit einem Zeichenfeld selbst Zahlen malen und vorhersagen lassen

## Was ist logistische Regression?

Im letzten Notebook haben wir **lineare Regression** kennengelernt: Damit haben wir eine **Zahl** vorhergesagt (z.B. den Preis eines LEGO-Sets).

Jetzt wollen wir eine andere Frage beantworten: **Gehört etwas zu Kategorie A oder Kategorie B?** Das nennt man **Klassifikation**.

Ein paar Beispiele:
- Ist diese E-Mail **Spam** oder **kein Spam**?
- Ist dieser Tumor **gutartig** oder **bösartig**?
- Zeigt dieses Bild eine **Katze** oder einen **Hund**?

Die **logistische Regression** ist ein Modell für genau solche Ja/Nein-Entscheidungen. Trotz des Namens "Regression" ist sie ein **Klassifikationsverfahren**.

### Die Sigmoid-Funktion (S-Kurve)

Die Idee: Statt eine beliebige Zahl vorherzusagen, berechnet die logistische Regression eine **Wahrscheinlichkeit** zwischen 0 und 1. Dazu nutzt sie die sogenannte **Sigmoid-Funktion**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Diese Funktion hat eine typische **S-Form**. Egal welchen Wert man einsetzt – das Ergebnis liegt immer zwischen 0 und 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Die Sigmoid-Funktion
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# S-Kurve plotten
z = np.linspace(-8, 8, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sigmoid(z), color="steelblue", linewidth=2.5)
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(x=0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("z (Eingabewert)")
ax.set_ylabel("σ(z) (Wahrscheinlichkeit)")
ax.set_title("Die Sigmoid-Funktion – typische S-Kurve")
ax.set_ylim(-0.05, 1.05)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])

# Beschriftungen
ax.annotate("Klasse 0\n(z.B. kein Spam)", xy=(-6, 0.05), fontsize=10, color="coral",
            ha="center", fontweight="bold")
ax.annotate("Klasse 1\n(z.B. Spam)", xy=(6, 0.95), fontsize=10, color="seagreen",
            ha="center", fontweight="bold")
ax.annotate("Schwelle: 0.5", xy=(0.3, 0.52), fontsize=9, color="gray")

plt.tight_layout()
plt.show()

print("Wenn σ(z) > 0.5 → Klasse 1 (z.B. Spam)")
print("Wenn σ(z) < 0.5 → Klasse 0 (z.B. kein Spam)")

### Von einer Variable zu vielen – und von 2 Klassen zu 10

Im Beispiel oben hatten wir **eine** Eingabevariable und **zwei** Klassen (Spam / kein Spam). Aber die logistische Regression kann viel mehr!

**Viele Eingabevariablen:** Statt nur einer Zahl können wir dem Modell **hunderte** Werte gleichzeitig geben. Ein Bild mit 28 × 28 Pixeln hat zum Beispiel **784 Pixel-Werte** – jeder Pixel ist eine Eingabevariable. Das Modell lernt dann, welche Pixel-Kombinationen für welche Ziffer typisch sind.

**Viele Klassen:** Mit einem Trick namens **One-vs-Rest** kann die logistische Regression auch mehr als zwei Klassen unterscheiden. Für 10 Ziffern (0–9) trainiert sie intern **10 kleine Modelle** – eines pro Ziffer. Jedes Modell beantwortet die Frage: *Ist es diese Ziffer oder nicht?* Am Ende gewinnt die Ziffer mit der höchsten Wahrscheinlichkeit.

Das bedeutet: **Mit logistischer Regression können wir Handschriften erkennen!**

### Ist logistische Regression der beste Classifier?

Ehrlich gesagt: **Nein.** Für Bilderkennung gibt es bessere Verfahren – zum Beispiel neuronale Netze (Deep Learning), die wir vielleicht in einem späteren Notebook kennenlernen.

Aber die logistische Regression hat große Vorteile:
- Sie ist **einfach zu verstehen** – kein Black-Box-Modell
- Sie trainiert **schnell** – auch auf einem normalen Laptop in wenigen Sekunden
- Sie liefert trotzdem **überraschend gute Ergebnisse** (über 90 % Genauigkeit auf MNIST!)
- Man kann dabei **viel lernen**: Train/Test-Split, Accuracy, Vorhersagen auswerten

Deshalb starten wir damit – und du wirst sehen, wie viel ein "einfaches" Modell schon kann.

---

## Teil 1: Der MNIST-Datensatz

**MNIST** (Modified National Institute of Standards and Technology) ist einer der berühmtesten Datensätze im Machine Learning. Er enthält **70.000 handgeschriebene Ziffern** (0–9), jeweils als kleines Graustufenbild mit 28 × 28 Pixeln.

Jedes Bild hat also 28 × 28 = **784 Pixel**. Jeder Pixel hat einen Wert zwischen 0 (schwarz) und 255 (weiß). Diese 784 Werte sind unsere **Eingabevariablen** (Features).

Wir laden den Datensatz direkt aus scikit-learn:

In [ ]:
from sklearn.datasets import fetch_openml

# MNIST-Datensatz laden (kann beim ersten Mal etwas dauern)
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")

X = mnist.data       # Bilder: 70.000 × 784
y = mnist.target      # Labels: 70.000 Ziffern (als Strings "0"–"9")

# Pixel-Werte normalisieren: durch 255 teilen, sodass alle Werte zwischen 0 und 1 liegen.
# Beispiel: ein Pixel mit Wert 128 wird zu 128/255 ≈ 0.50.
# Das hilft dem Modell, schneller und besser zu lernen.
X = X / 255.0

print(f"Bilder:  {X.shape}  (70.000 Bilder mit je 784 Pixel-Werten)")
print(f"Labels:  {y.shape}  (die zugehörige Ziffer für jedes Bild)")
print(f"Wertebereich: {X.min():.2f} (schwarz) bis {X.max():.2f} (weiß)")

### So sehen die Daten aus

Jedes Bild ist ein **Vektor mit 784 Zahlen** – ein flaches Array. Um es als Bild anzuzeigen, bringen wir es zurück in die Form 28 × 28:

In [ ]:
# 20 Beispielbilder anzeigen
fig, axes = plt.subplots(2, 10, figsize=(12, 3))

for idx, ax in enumerate(axes.flat):
    bild = X[idx].reshape(28, 28)
    ax.imshow(bild, cmap="gray", vmin=0, vmax=1)
    ax.set_title(y[idx], fontsize=11)
    ax.axis("off")

fig.suptitle("Beispiele aus dem MNIST-Datensatz", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Jedes Bild hat {X.shape[1]} Pixel-Werte (28 × 28 = 784)")
print(f"Wertebereich: {X.min():.0f} (schwarz) bis {X.max():.0f} (weiß)")

---

## Teil 2: Train/Test-Split – Faire Bewertung

Bevor wir ein Modell trainieren, müssen wir einen wichtigen Schritt machen: Die Daten **aufteilen**.

Warum? Stell dir vor, du lernst für eine Prüfung und übst nur mit den Aufgaben, die dann auch in der Prüfung drankommen. Natürlich schaffst du dann 100 % – aber hast du den Stoff wirklich **verstanden**?

Genauso ist es beim Machine Learning:
- **Trainingsdaten** – damit lernt das Modell (wie Übungsaufgaben)
- **Testdaten** – damit prüfen wir, wie gut das Modell **neue, ungesehene** Daten erkennt (wie die Prüfung)

Wir halten also einen Teil der Daten **zurück** und zeigen sie dem Modell beim Training **nicht**. Erst danach testen wir damit.

Die Funktion `train_test_split` aus scikit-learn macht das automatisch:

In [ ]:
from sklearn.model_selection import train_test_split

# 80 % Training, 20 % Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Trainingsdaten: {X_train.shape[0]} Bilder")
print(f"Testdaten:      {X_test.shape[0]} Bilder")
print(f"\nDas Modell lernt mit {X_train.shape[0]} Bildern.")
print(f"Danach prüfen wir mit {X_test.shape[0]} Bildern, die es noch nie gesehen hat.")

---

## Zusammenfassung & Ausblick

Das haben wir bisher gelernt:

- Die **logistische Regression** berechnet Wahrscheinlichkeiten mit der **Sigmoid-Funktion**
- Sie kann nicht nur 2, sondern auch **10 Klassen** unterscheiden (One-vs-Rest)
- Statt einer Variable kann sie **784 Pixel-Werte** gleichzeitig verarbeiten
- Ein fairer Test braucht einen **Train/Test-Split**: Training und Prüfung mit verschiedenen Daten

Jetzt bist du dran: Trainiere das Modell und finde heraus, wie gut es Ziffern erkennen kann!

---

# Übungsaufgaben – Logistische Regression mit MNIST

Jetzt bist du dran! Trainiere einen Classifier, der handgeschriebene Ziffern erkennt.

Wir nutzen `LogisticRegression` aus scikit-learn – das funktioniert genauso wie `LinearRegression` aus dem letzten Notebook:

```python
from sklearn.linear_model import LogisticRegression

modell = LogisticRegression(max_iter=1000)
modell.fit(X_train, y_train)           # Modell trainieren
vorhersagen = modell.predict(X_test)   # Vorhersagen berechnen
```

Der Parameter `max_iter=1000` gibt dem Modell genug Rechenzeit, um die beste Lösung zu finden.

### Aufgabe 1: Modell trainieren und Vorhersagen berechnen

Trainiere eine logistische Regression auf den **Trainingsdaten** und berechne dann **Vorhersagen** für die Testdaten.

**Schritt für Schritt:**

1. Erstelle ein `LogisticRegression`-Modell mit `max_iter=1000`.
2. Trainiere das Modell mit `.fit()` auf `X_train` und `y_train`.
3. Berechne Vorhersagen für die Testdaten mit `.predict()` auf `X_test`.
4. Zeige die ersten 20 Vorhersagen und vergleiche sie mit den echten Labels.

*Hinweis:* Das Training kann 1–2 Minuten dauern – das ist normal bei 56.000 Bildern!

In [ ]:
# Aufgabe 1: Modell trainieren und Vorhersagen berechnen

from sklearn.linear_model import LogisticRegression

# 1. Modell erstellen
modell = LogisticRegression(max_iter=1000)

# 2. Modell trainieren (das dauert ca. 1–2 Minuten)
print("Training läuft...")
modell.fit(X_train, y_train)
print("Training abgeschlossen!")

# 3. Vorhersagen für die Testdaten berechnen
vorhersagen = modell.predict(X_test)

# 4. Die ersten 20 Vorhersagen mit den echten Labels vergleichen
print(f"\nVorhersage: {list(vorhersagen[:20])}")
print(f"Echt:       {list(y_test[:20])}")

### Aufgabe 2: Wie gut ist das Modell? – Accuracy berechnen

Die **Accuracy** (Genauigkeit) gibt an, welcher Anteil der Vorhersagen **richtig** war:

$$\text{Accuracy} = \frac{\text{Anzahl richtiger Vorhersagen}}{\text{Gesamtanzahl Vorhersagen}}$$

Ein Wert von 0.92 bedeutet zum Beispiel: 92 % der Ziffern wurden korrekt erkannt.

**Aufgabe:**

1. Zähle, wie viele Vorhersagen richtig waren. *Tipp:* `(vorhersagen == y_test).sum()`
2. Berechne daraus die Accuracy.
3. Wie viele der 14.000 Testbilder hat das Modell falsch erkannt?

In [ ]:
# Aufgabe 2: Accuracy berechnen

# 1. Wie viele Vorhersagen waren richtig?
richtig = (vorhersagen == y_test).sum()
print(f"Richtig erkannt: {richtig} von {len(y_test)}")

# 2. Accuracy berechnen
accuracy = richtig / len(y_test)
print(f"Accuracy: {accuracy:.4f}")

# 3. Wie viele Bilder wurden falsch erkannt?
falsch = len(y_test) - richtig
print(f"Falsch erkannt: {falsch} Bilder")

### Gut zu wissen: `accuracy_score` aus scikit-learn

In der Praxis muss man die Accuracy nicht jedes Mal von Hand berechnen. scikit-learn hat dafür eine fertige Funktion – `accuracy_score`. Sie macht genau das Gleiche wie unsere Berechnung oben:

In [ ]:
from sklearn.metrics import accuracy_score

print(f"Accuracy (sklearn): {accuracy_score(y_test, vorhersagen):.4f}")

---

## Teil 3: Eigene Ziffern erkennen

Jetzt wird es spannend! Wir bauen ein **interaktives Zeichenfeld**, in dem du mit der Maus eine Ziffer malen kannst. Das Bild wird auf 28 × 28 Pixel herunterskaliert – genau wie die MNIST-Bilder. Dann lassen wir unser trainiertes Modell vorhersagen, welche Ziffer du gemalt hast.

### Das Zeichenfeld

Das Zeichenfeld ist 280 × 280 Pixel groß (10x so groß wie MNIST), damit man bequem zeichnen kann. Beim Klick auf "Übernehmen" wird das Bild automatisch auf 28 × 28 verkleinert und invertiert (MNIST hat helle Schrift auf dunklem Grund).

**Hinweis:** Der Code in der nächsten Zelle ist etwas aufwendiger als das, was wir bisher geschrieben haben – ihr müsst hier nicht jeden Schritt verstehen. Wichtig ist nur: Zelle ausführen, Ziffer malen, auf "Übernehmen" klicken!

In [ ]:
%matplotlib widget
from matplotlib.widgets import Button as MplButton
from PIL import Image

# ── Zeichenfeld ──

# Globale Variable: hier landet das fertige 28x28-Bild
ziffer_bild = None

# 280x280 Pixel-Leinwand (weiß)
leinwand = np.ones((280, 280), dtype=np.uint8) * 255
stiftbreite = 8  # Radius in Pixeln

# --- Figure aufbauen ---
fig_draw = plt.figure(figsize=(5, 5.8))

# Zeichenfläche (oben)
ax_draw = fig_draw.add_axes([0.1, 0.18, 0.8, 0.75])
bild_anzeige = ax_draw.imshow(leinwand, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
ax_draw.set_title("Male eine Ziffer (0–9) mit der Maus")
ax_draw.set_xticks([])
ax_draw.set_yticks([])

# Buttons (unten)
ax_btn_clear = fig_draw.add_axes([0.15, 0.04, 0.3, 0.06])
ax_btn_submit = fig_draw.add_axes([0.55, 0.04, 0.3, 0.06])
btn_clear = MplButton(ax_btn_clear, "Löschen")
btn_submit = MplButton(ax_btn_submit, "Übernehmen")

# --- Zeichen-Logik ---
zeichnet = False
letzte_pos = None

def strich_malen(x, y):
    """Malt einen ausgefüllten Kreis an Position (x, y)."""
    for dy in range(-stiftbreite, stiftbreite + 1):
        for dx in range(-stiftbreite, stiftbreite + 1):
            if dx * dx + dy * dy <= stiftbreite ** 2:
                px, py = x + dx, y + dy
                if 0 <= px < 280 and 0 <= py < 280:
                    leinwand[py, px] = 0

def linie_malen(x0, y0, x1, y1):
    """Malt eine Linie von (x0,y0) nach (x1,y1) durch viele kleine Kreise."""
    schritte = max(abs(x1 - x0), abs(y1 - y0), 1)
    for i in range(schritte + 1):
        t = i / schritte
        strich_malen(int(x0 + t * (x1 - x0)), int(y0 + t * (y1 - y0)))

def on_press(event):
    global zeichnet, letzte_pos
    if event.inaxes != ax_draw:
        return
    zeichnet = True
    x, y = int(event.xdata), int(event.ydata)
    letzte_pos = (x, y)
    strich_malen(x, y)
    bild_anzeige.set_data(leinwand)
    fig_draw.canvas.draw_idle()

def on_move(event):
    global letzte_pos
    if not zeichnet or event.inaxes != ax_draw:
        return
    x, y = int(event.xdata), int(event.ydata)
    if letzte_pos is not None:
        linie_malen(letzte_pos[0], letzte_pos[1], x, y)
    letzte_pos = (x, y)
    bild_anzeige.set_data(leinwand)
    fig_draw.canvas.draw_idle()

def on_release(event):
    global zeichnet, letzte_pos
    zeichnet = False
    letzte_pos = None

def on_clear(event):
    leinwand[:] = 255
    bild_anzeige.set_data(leinwand)
    fig_draw.canvas.draw_idle()

def on_submit(event):
    global ziffer_bild
    bild = Image.fromarray(leinwand, mode="L")
    bild_28 = bild.resize((28, 28), Image.LANCZOS)
    ziffer_bild = 255 - np.array(bild_28)
    print(f"\nBild übernommen! Form: {ziffer_bild.shape}, Werte: {ziffer_bild.min()}–{ziffer_bild.max()}")

fig_draw.canvas.mpl_connect("button_press_event", on_press)
fig_draw.canvas.mpl_connect("motion_notify_event", on_move)
fig_draw.canvas.mpl_connect("button_release_event", on_release)
btn_clear.on_clicked(on_clear)
btn_submit.on_clicked(on_submit)

plt.show()

### Vorschau des 28x28-Bildes

Nachdem du eine Ziffer gemalt und auf **Übernehmen** geklickt hast, kannst du dir das herunterskalierte 28 × 28-Bild anschauen. So sieht es aus, wenn es an den Classifier übergeben wird:

In [ ]:
# Vorschau: So sieht das Bild im MNIST-Format aus

if ziffer_bild is None:
    print("Noch kein Bild übernommen! Male oben eine Ziffer und klicke 'Übernehmen'.")
else:
    fig2, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(ziffer_bild, cmap="gray", vmin=0, vmax=255)
    ax.set_title("28x28 Pixel (MNIST-Format)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Bildgröße:    {ziffer_bild.shape}")
    print(f"Wertebereich: {ziffer_bild.min()} – {ziffer_bild.max()}")

### Vorhersage mit dem trainierten Modell

Jetzt lassen wir unser Modell die gemalte Ziffer erkennen! Das Bild wird als **flacher Vektor** (784 Werte) an das Modell übergeben – genau wie die MNIST-Trainingsdaten.

In [ ]:
# Vorhersage für die selbst gemalte Ziffer

if ziffer_bild is None:
    print("Noch kein Bild übernommen! Male oben eine Ziffer und klicke 'Übernehmen'.")
else:
    # Bild als flachen Vektor (1 × 784) vorbereiten und normalisieren (0–1)
    bild_vektor = ziffer_bild.reshape(1, -1).astype(float) / 255.0

    # Vorhersage
    vorhersage = modell.predict(bild_vektor)[0]

    # Wahrscheinlichkeiten für alle 10 Klassen
    wahrscheinlichkeiten = modell.predict_proba(bild_vektor)[0]

    # Ergebnis anzeigen
    fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                     gridspec_kw={"width_ratios": [1, 2]})

    ax1.imshow(ziffer_bild, cmap="gray", vmin=0, vmax=255)
    ax1.set_title(f"Vorhersage: {vorhersage}", fontsize=16, fontweight="bold")
    ax1.axis("off")

    farben = ["steelblue"] * 10
    farben[int(vorhersage)] = "darkorange"
    ax2.bar(range(10), wahrscheinlichkeiten * 100, color=farben, edgecolor="white")
    ax2.set_xlabel("Ziffer")
    ax2.set_ylabel("Wahrscheinlichkeit (%)")
    ax2.set_title("Wahrscheinlichkeiten pro Klasse")
    ax2.set_xticks(range(10))
    ax2.set_ylim(0, 105)

    fig3.tight_layout()
    plt.show()

    print(f"Das Modell erkennt: {vorhersage}")
    print(f"Sicherheit: {wahrscheinlichkeiten[int(vorhersage)] * 100:.1f} %")

---

## Geschafft!

Du hast heute gelernt:

- Was die **logistische Regression** ist und wie die **Sigmoid-Funktion** funktioniert
- Dass man mit logistischer Regression auch **viele Klassen** und **viele Variablen** gleichzeitig verarbeiten kann
- Wie man Daten in **Training** und **Test** aufteilt, um ein Modell fair zu bewerten
- Wie man mit scikit-learn ein Modell **trainiert** und **Vorhersagen** berechnet
- Was **Accuracy** bedeutet und wie man sie berechnet
- Wie man mit einem trainierten Modell **eigene handgeschriebene Ziffern** erkennen kann

Die logistische Regression erreicht auf MNIST über 90 % Genauigkeit – und das ganz ohne neuronale Netze! In einem späteren Notebook schauen wir uns an, wie man mit komplexeren Modellen noch besser werden kann.

---

### Experten-Aufgabe: Eigene Ziffern besser erkennen

Dir ist vielleicht aufgefallen, dass das Modell bei selbst gemalten Ziffern manchmal daneben liegt, obwohl es auf den MNIST-Testdaten über 90 % schafft. Woran liegt das?

Das Problem: Die MNIST-Bilder sind **vorverarbeitet**. Jede Ziffer ist **zentriert** und nimmt ungefähr **20 × 20 Pixel** des 28 × 28-Bildes ein (ca. 70 % der Seitenlänge). Wenn du im Zeichenfeld aber ganz klein in eine Ecke malst oder das ganze Feld ausfüllst, sieht das Bild ganz anders aus als die Trainingsdaten – und das Modell wird unsicher.

**Deine Aufgabe:** Schreibe eine Funktion `bild_zentrieren(bild_28)`, die ein 28 × 28-Bild so aufbereitet, dass die Ziffer zentriert und auf die richtige Größe skaliert ist.

**Tipps:**

1. **Bounding Box finden:** Suche die Zeilen und Spalten, in denen tatsächlich Pixel gesetzt sind (Wert > 0). `np.where(bild_28 > 0)` gibt dir die Koordinaten aller nicht-schwarzen Pixel.
2. **Ausschneiden:** Schneide den Bereich mit der Ziffer aus dem Bild aus.
3. **Skalieren:** Verkleinere den Ausschnitt auf ca. 20 × 20 Pixel (mit `Image.resize` aus PIL).
4. **Zentrieren:** Setze den skalierten Ausschnitt in die Mitte eines neuen 28 × 28-Bildes (das komplett schwarz ist).

```python
def bild_zentrieren(bild_28):
    # Schritt 1: Wo sind Pixel gesetzt?
    zeilen, spalten = np.where(bild_28 > 0)

    # Schritt 2: Bounding Box ausschneiden
    ausschnitt = bild_28[zeilen.min():zeilen.max()+1, spalten.min():spalten.max()+1]

    # Schritt 3: Auf 20x20 skalieren
    # ... (mit PIL)

    # Schritt 4: In 28x28 zentriert einsetzen
    # ... (mit np.zeros)

    return neues_bild
```

In [ ]:
# Experten-Aufgabe: bild_zentrieren() implementieren

# TODO: Funktion schreiben und mit ziffer_bild testen